# Test: Qwen3-235B-A22B-Instruct-2507

Primary agents + Host — GPU 0-1 (NVLink), Port 8000

**Prerequisites:** vLLM server running on port 8000

In [3]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))  # project root

from models.utils import Qwen3_235B

model = Qwen3_235B()
print('Model config:')
model.get_config()

Model config:


{'model_id': 'Qwen/Qwen3-235B-A22B-Instruct-2507',
 'model_path': '/storage/data/AgenticCyOps_Private/models/Qwen/Qwen3-235B-A22B-Instruct-2507',
 'role': 'primary_agents_and_host',
 'architecture': 'MoE',
 'total_params': '235B',
 'active_params': '22B',
 'gpu_assignment': '0,1',
 'port': 8000,
 'base_url': 'http://localhost:8000/v1'}

## 1. Health Check

In [4]:
assert model.health_check(), 'Server not running on port 8000!'
print('Health check passed')

Health check passed


## 2. List Models

In [5]:
models = model.list_models()
for m in models:
    print(f'  {m.id}')

  /storage/data/AgenticCyOps_Private/models/Qwen/Qwen3-235B-A22B-Instruct-2507


## 3. Chat Completions

In [6]:
# 3a. Basic chat
messages = [
    {'role': 'system', 'content': 'You are a SOC analyst. Be concise.'},
    {'role': 'user', 'content': 'What is a lateral movement attack? One sentence.'}
]
resp = model.chat(messages, max_tokens=100)
print('Basic chat:', resp.choices[0].message.content)

Basic chat: A lateral movement attack is when an attacker gains initial access to a system and then moves through the network to escalate privileges, access other systems, and expand control.


In [7]:
# 3b. Deterministic (temp=0.0)
resp1 = model.chat_deterministic(messages, max_tokens=100)
resp2 = model.chat_deterministic(messages, max_tokens=100)
print('Deterministic r1:', resp1.choices[0].message.content[:80])
print('Deterministic r2:', resp2.choices[0].message.content[:80])
print('Match:', resp1.choices[0].message.content == resp2.choices[0].message.content)

Deterministic r1: A lateral movement attack is when an attacker gains initial access to a system a
Deterministic r2: A lateral movement attack is when an attacker gains initial access to a system a
Match: True


In [8]:
# 3c. Creative (temp=0.7)
resp = model.chat_creative(messages, max_tokens=100)
print('Creative:', resp.choices[0].message.content)

Creative: A lateral movement attack is when an attacker gains initial access to one system and then moves across a network to compromise additional systems and escalate privileges.


In [9]:
# 3d. Streaming
stream = model.chat(messages, max_tokens=100, stream=True)
print('Streaming: ', end='')
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end='', flush=True)
print()

Streaming: A lateral movement attack is when an attacker gains initial access to a system and then moves through the network to escalate privileges, access other systems, and expand control.


In [10]:
# 3e. Custom parameters (top_p, frequency_penalty, stop sequences)
resp = model.chat(
    messages, max_tokens=200,
    top_p=0.9, top_k=50,
    frequency_penalty=0.5, presence_penalty=0.3,
    stop=['\n\n']
)
print('Custom params:', resp.choices[0].message.content)

Custom params: A lateral movement attack is when an attacker gains initial access to a system and then moves through the network to escalate privileges, access other systems, and expand control.


## 4. Tool / Function Calling

In [11]:
tools = [
    {
        'type': 'function',
        'function': {
            'name': 'query_siem',
            'description': 'Search SIEM logs for security events',
            'parameters': {
                'type': 'object',
                'properties': {
                    'query': {'type': 'string', 'description': 'Search query'},
                    'time_range': {'type': 'string', 'description': 'Time range (e.g. last_24h)'},
                },
                'required': ['query']
            }
        }
    },
    {
        'type': 'function',
        'function': {
            'name': 'isolate_host',
            'description': 'Isolate a host from the network',
            'parameters': {
                'type': 'object',
                'properties': {
                    'hostname': {'type': 'string', 'description': 'Host to isolate'},
                    'reason': {'type': 'string', 'description': 'Reason for isolation'},
                },
                'required': ['hostname', 'reason']
            }
        }
    }
]
print(f'Defined {len(tools)} tools')

Defined 2 tools


In [12]:
# 4a. Auto tool choice
tc_messages = [
    {'role': 'system', 'content': 'You are a SOC analyst with access to SIEM and host isolation tools.'},
    {'role': 'user', 'content': 'Check SIEM for failed logins from 10.0.5.12 in the last hour.'}
]
resp = model.tool_call(tc_messages, tools)
tc = resp.choices[0].message.tool_calls
print(f'Auto tool choice: {len(tc)} call(s)')
for c in tc:
    print(f'  {c.function.name}({c.function.arguments})')

Auto tool choice: 1 call(s)
  query_siem({"query": "source_ip:10.0.5.12 AND event_type:failed_login", "time_range": "last_1h"})


In [13]:
# 4b. Required tool choice
resp = model.tool_call_required(tc_messages, tools)
tc = resp.choices[0].message.tool_calls
print(f'Required: {len(tc)} call(s)')
for c in tc:
    print(f'  {c.function.name}({c.function.arguments})')

Required: 1 call(s)
  query_siem({"query": "source_ip:10.0.5.12 AND event_type:failed_login", "time_range": "last_1h"})


In [14]:
# 4c. Specific tool choice
resp = model.tool_call_specific(tc_messages, tools, 'isolate_host')
tc = resp.choices[0].message.tool_calls
print(f'Specific (isolate_host): {len(tc)} call(s)')
for c in tc:
    print(f'  {c.function.name}({c.function.arguments})')

Specific (isolate_host): 1 call(s)
  isolate_host({"hostname": "web-server-01", "reason": "malware infection"})


## 5. Structured Output

In [15]:
# 5a. JSON mode
json_messages = [
    {'role': 'system', 'content': 'Respond with JSON only.'},
    {'role': 'user', 'content': 'Classify this alert: "Multiple failed SSH logins from 10.0.5.12". Return {"severity": str, "category": str, "confidence": float}'}
]
resp = model.chat_json(json_messages, temperature=0.0, max_tokens=200)
print('JSON mode:', resp.choices[0].message.content)

JSON mode: {"severity": "high", "category": "brute_force", "confidence": 0.95}


In [16]:
# 5b. JSON schema
schema = {
    'type': 'object',
    'properties': {
        'severity': {'type': 'string', 'enum': ['low', 'medium', 'high', 'critical']},
        'category': {'type': 'string'},
        'confidence': {'type': 'number', 'minimum': 0, 'maximum': 1},
        'iocs': {'type': 'array', 'items': {'type': 'string'}}
    },
    'required': ['severity', 'category', 'confidence']
}
resp = model.chat_json_schema(json_messages, schema, schema_name='alert_classification', temperature=0.0, max_tokens=300)
print('JSON schema:', resp.choices[0].message.content)

JSON schema: {"severity": "high", "category": "brute_force", "confidence": 0.95}


## 6. Batch Inference

In [17]:
batches = [
    [{'role': 'user', 'content': 'What is phishing? One sentence.'}],
    [{'role': 'user', 'content': 'What is ransomware? One sentence.'}],
    [{'role': 'user', 'content': 'What is a zero-day? One sentence.'}],
]
results = model.batch_chat(batches, max_tokens=80)
for i, r in enumerate(results):
    print(f'Batch {i}: {r.choices[0].message.content}')

Batch 0: Phishing is a cyber attack where attackers impersonate legitimate organizations to trick individuals into revealing sensitive information such as passwords, credit card numbers, or personal data.
Batch 1: Ransomware is a type of malicious software that encrypts a victim's files or locks their system, demanding payment in exchange for restoring access.
Batch 2: A zero-day is a previously unknown software vulnerability that is exploited by attackers before the developer has released a patch to fix it.


## 7. Token Usage Tracking

In [18]:
resp = model.chat(messages, max_tokens=100)
usage = resp.usage
print(f'Prompt tokens:     {usage.prompt_tokens}')
print(f'Completion tokens: {usage.completion_tokens}')
print(f'Total tokens:      {usage.total_tokens}')

Prompt tokens:     32
Completion tokens: 33
Total tokens:      65


## 8. Get Config

In [19]:
import json
config = model.get_config()
print(json.dumps(config, indent=2))

{
  "model_id": "Qwen/Qwen3-235B-A22B-Instruct-2507",
  "model_path": "/storage/data/AgenticCyOps_Private/models/Qwen/Qwen3-235B-A22B-Instruct-2507",
  "role": "primary_agents_and_host",
  "architecture": "MoE",
  "total_params": "235B",
  "active_params": "22B",
  "gpu_assignment": "0,1",
  "port": 8000,
  "base_url": "http://localhost:8000/v1"
}


## Summary

All tests passed if no cells raised exceptions above.